In [1]:
# Cell 1: Mount Drive and load multi-stock news data
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = '/content/drive/MyDrive/financial-news-sentiment-stock-predictor'
DATA_PATH = f'{PROJECT_PATH}/data'

import pandas as pd

news_data = pd.read_csv(f'{DATA_PATH}/news_data.csv')
news_data['date'] = pd.to_datetime(news_data['date'])

print(f"Loaded news data: {news_data.shape}")
print(f"\nArticles per ticker:")
print(news_data['ticker'].value_counts().sort_index())
print(f"\nDate range: {news_data['date'].min().date()} "
      f"to {news_data['date'].max().date()}")

Mounted at /content/drive
Loaded news data: (712, 5)

Articles per ticker:
ticker
AAPL     164
AMZN     107
GOOGL    137
MSFT     151
TSLA     153
Name: count, dtype: int64

Date range: 2026-05-17 to 2026-06-14


In [2]:
# Cell 2: Install transformers and load FinBERT pipeline
!pip install transformers torch -q

from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    device=-1  # CPU; change to 0 if GPU available
)

print("FinBERT loaded successfully")

# Quick sanity check
test = sentiment_pipeline(
    ["Apple stock surged after strong earnings beat",
     "Microsoft shares fell amid cloud slowdown concerns",
     "Tesla deliveries missed analyst expectations badly"]
)
for t, r in zip(
    ["AAPL positive", "MSFT negative", "TSLA negative"], test):
    print(f"  {t}: {r}")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

FinBERT loaded successfully
  AAPL positive: {'label': 'positive', 'score': 0.9407472014427185}
  MSFT negative: {'label': 'negative', 'score': 0.9638744592666626}
  TSLA negative: {'label': 'negative', 'score': 0.9467268586158752}


In [3]:
# Cell 3: Run FinBERT sentiment analysis on all 712 headlines
headlines = news_data['headline'].tolist()

print(f"Running FinBERT on {len(headlines)} headlines...")
results = sentiment_pipeline(
    headlines,
    batch_size=16,
    truncation=True
)

# Add raw results to DataFrame
news_data['sentiment_label'] = [r['label'] for r in results]
news_data['sentiment_score_raw'] = [r['score'] for r in results]

# Convert to signed sentiment score
def to_signed_score(row):
    if row['sentiment_label'] == 'positive':
        return row['sentiment_score_raw']
    elif row['sentiment_label'] == 'negative':
        return -row['sentiment_score_raw']
    else:
        return 0.0

news_data['sentiment_score'] = news_data.apply(
    to_signed_score, axis=1)

print("FinBERT inference complete")
print(f"\nSentiment distribution across all tickers:")
print(news_data['sentiment_label'].value_counts())
print(f"\nPer ticker sentiment breakdown:")
for ticker in ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']:
    subset = news_data[news_data['ticker'] == ticker]
    counts = subset['sentiment_label'].value_counts()
    print(f"  {ticker}: "
          f"pos={counts.get('positive', 0)} | "
          f"neg={counts.get('negative', 0)} | "
          f"neu={counts.get('neutral', 0)}")

Running FinBERT on 712 headlines...
FinBERT inference complete

Sentiment distribution across all tickers:
sentiment_label
neutral     471
negative    126
positive    115
Name: count, dtype: int64

Per ticker sentiment breakdown:
  AAPL: pos=27 | neg=18 | neu=119
  MSFT: pos=15 | neg=27 | neu=109
  GOOGL: pos=22 | neg=22 | neu=93
  AMZN: pos=23 | neg=28 | neu=56
  TSLA: pos=28 | neg=31 | neu=94


In [4]:
# Cell 4: Aggregate sentiment scores at ticker-day level
daily_sentiment = news_data.groupby(
    ['date', 'ticker']).agg(
    avg_sentiment=('sentiment_score', 'mean'),
    sentiment_std=('sentiment_score', 'std'),
    news_count=('sentiment_score', 'count'),
    positive_count=('sentiment_label',
                    lambda x: (x == 'positive').sum()),
    negative_count=('sentiment_label',
                    lambda x: (x == 'negative').sum())
).reset_index()

# Fill NaN std (single headline days)
daily_sentiment['sentiment_std'] = \
    daily_sentiment['sentiment_std'].fillna(0)

print(f"Daily sentiment shape: {daily_sentiment.shape}")
print(f"\nRows per ticker:")
print(daily_sentiment['ticker'].value_counts().sort_index())

print(f"\nSentiment statistics per ticker:")
for ticker in ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']:
    subset = daily_sentiment[
        daily_sentiment['ticker'] == ticker]
    print(f"\n  {ticker}:")
    print(f"    Days with news: {len(subset)}")
    print(f"    Avg sentiment: "
          f"{subset['avg_sentiment'].mean():.4f}")
    print(f"    Avg news/day: "
          f"{subset['news_count'].mean():.1f}")
    print(f"    Most active day: "
          f"{subset.loc[subset['news_count'].idxmax(), 'date'].date()} "
          f"({subset['news_count'].max()} articles)")

daily_sentiment.head(10)

Daily sentiment shape: (121, 7)

Rows per ticker:
ticker
AAPL     24
AMZN     24
GOOGL    26
MSFT     19
TSLA     28
Name: count, dtype: int64

Sentiment statistics per ticker:

  AAPL:
    Days with news: 24
    Avg sentiment: 0.0101
    Avg news/day: 6.8
    Most active day: 2026-06-08 (41 articles)

  MSFT:
    Days with news: 19
    Avg sentiment: 0.0012
    Avg news/day: 7.9
    Most active day: 2026-06-10 (20 articles)

  GOOGL:
    Days with news: 26
    Avg sentiment: 0.0085
    Avg news/day: 5.3
    Most active day: 2026-06-01 (10 articles)

  AMZN:
    Days with news: 24
    Avg sentiment: -0.0736
    Avg news/day: 4.5
    Most active day: 2026-06-13 (14 articles)

  TSLA:
    Days with news: 28
    Avg sentiment: -0.0070
    Avg news/day: 5.5
    Most active day: 2026-06-12 (20 articles)


,date,ticker,avg_sentiment,sentiment_std,news_count,positive_count,negative_count
0,2026-05-17,AMZN,-0.940919,0.000000,1,0,1
1,2026-05-17,GOOGL,0.000000,0.000000,1,0,0
2,2026-05-17,TSLA,0.912892,0.000000,1,1,0
3,2026-05-18,AMZN,-0.217124,0.385083,7,0,2
4,2026-05-18,GOOGL,0.750296,0.000000,1,1,0
5,2026-05-18,TSLA,0.332421,1.005406,3,2,1
6,2026-05-19,AAPL,0.000000,0.000000,1,0,0
7,2026-05-19,AMZN,-0.312965,0.542071,3,0,1
8,2026-05-19,GOOGL,0.000000,0.000000,3,0,0
9,2026-05-19,TSLA,-0.242897,0.366664,9,0,3


In [5]:
# Cell 5: Final validation and save sentiment features
print("Null check:")
print(daily_sentiment.isnull().sum())

print(f"\nDate range: {daily_sentiment['date'].min().date()} "
      f"to {daily_sentiment['date'].max().date()}")

print(f"\nTotal ticker-day rows: {len(daily_sentiment)}")
print(f"Expected coverage in final dataset:")
for ticker in ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']:
    days_with_news = len(daily_sentiment[
        daily_sentiment['ticker'] == ticker])
    print(f"  {ticker}: {days_with_news} real sentiment days "
          f"vs ~1255 total trading days "
          f"({days_with_news/1255*100:.1f}% coverage)")

# Save
daily_sentiment.to_csv(
    f'{DATA_PATH}/sentiment_features.csv', index=False)
print(f"\nSaved to {DATA_PATH}/sentiment_features.csv")

Null check:
date              0
ticker            0
avg_sentiment     0
sentiment_std     0
news_count        0
positive_count    0
negative_count    0
dtype: int64

Date range: 2026-05-17 to 2026-06-14

Total ticker-day rows: 121
Expected coverage in final dataset:
  AAPL: 24 real sentiment days vs ~1255 total trading days (1.9% coverage)
  MSFT: 19 real sentiment days vs ~1255 total trading days (1.5% coverage)
  GOOGL: 26 real sentiment days vs ~1255 total trading days (2.1% coverage)
  AMZN: 24 real sentiment days vs ~1255 total trading days (1.9% coverage)
  TSLA: 28 real sentiment days vs ~1255 total trading days (2.2% coverage)

Saved to /content/drive/MyDrive/financial-news-sentiment-stock-predictor/data/sentiment_features.csv
